In [ ]:
!pip install transformers seqeval evaluate accelerate -U

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import numpy as np
import evaluate
seqeval = evaluate.load("seqeval") # Khai báo seqeval ở đây
def compute_metrics(p):
    predictions, labels = p

    # Biến ma trận xác suất (logits) thành con số ID nhãn dự đoán cao nhất
    predictions = np.argmax(predictions, axis=2)

    # Lọc bỏ các nhãn -100 ra khỏi quá trình tính toán metric
    true_predictions = [
        [id2label[p] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]
    true_labels = [
        [id2label[l] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]

    results = seqeval.compute(predictions=true_predictions, references=true_labels)
    return {
        "precision": results["overall_precision"],
        "recall": results["overall_recall"],
        "f1": results["overall_f1"],
        "accuracy": results["overall_accuracy"],
    }

In [ ]:
def load_conll_data(file_path):
    sentences = []
    current_sentence = []

    with open(file_path, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if not line:
                if current_sentence:
                    sentences.append(current_sentence)
                    current_sentence = []
            else:
                parts = line.split()
                if len(parts) >= 2:
                    word, tag = parts[0], parts[1]
                    current_sentence.append((word, tag))
        if current_sentence:
            sentences.append(current_sentence)
    return sentences

# Đọc tập train, dev
train_sentences = load_conll_data('/content/drive/MyDrive/datasetViMedNER/traindata/train.txt')
dev_sentences = load_conll_data('/content/drive/MyDrive/datasetViMedNER/traindata/dev.txt')

In [ ]:
# Đọc danh sách nhãn từ file labels.txt có sẵn trong thư mục train_dataset
label_path = '/content/drive/MyDrive/datasetViMedNER/traindata/labels.txt'

with open(label_path, 'r', encoding='utf-8') as f:
    unique_tags = [line.strip() for line in f if line.strip()]

# Tạo lại từ điển ánh xạ
label2id = {tag: idx for idx, tag in enumerate(unique_tags)}
id2label = {idx: tag for idx, tag in enumerate(unique_tags)}

print(f"✅ Đã tải thành công {len(label2id)} nhãn từ file labels.txt!")

✅ Đã tải thành công 11 nhãn từ file labels.txt!


In [ ]:
import torch
from torch.utils.data import Dataset
from transformers import AutoTokenizer

class ViMedNERDataset(Dataset):
    def __init__(self, sentences, tokenizer, label2id, max_length=128):
        self.sentences = sentences
        self.tokenizer = tokenizer
        self.label2id = label2id
        self.max_length = max_length

    def __len__(self):
        return len(self.sentences)

    def __getitem__(self, idx):
        sentence_data = self.sentences[idx]
        words = [item[0] for item in sentence_data]
        tags = [item[1] for item in sentence_data]

        # Tokenize từng từ và theo dõi số lượng sub-token của mỗi từ
        tokenized_outputs = []
        label_ids = []

        # Thêm token [CLS] đầu câu
        tokenized_outputs.append(self.tokenizer.cls_token_id)
        label_ids.append(-100)

        for word, tag in zip(words, tags):
            # Tokenize từng từ lẻ (không dùng is_split_into_words để tránh lỗi word_ids)
            sub_tokens = self.tokenizer.tokenize(word)
            sub_token_ids = self.tokenizer.convert_tokens_to_ids(sub_tokens)

            if len(sub_token_ids) > 0:
                # Sub-token đầu tiên nhận nhãn thật
                tokenized_outputs.append(sub_token_ids[0])
                label_ids.append(self.label2id[tag])

                # Các sub-token phía sau của cùng 1 từ nhận -100
                for sub_id in sub_token_ids[1:]:
                    tokenized_outputs.append(sub_id)
                    label_ids.append(-100)

        # Thêm token [SEP] cuối câu
        tokenized_outputs.append(self.tokenizer.sep_token_id)
        label_ids.append(-100)

        # Cắt ngắn (Truncation) nếu vượt quá max_length
        if len(tokenized_outputs) > self.max_length:
            tokenized_outputs = tokenized_outputs[:self.max_length]
            label_ids = label_ids[:self.max_length]

        # Tạo attention mask (1 cho token thật, 0 cho padding)
        attention_mask = [1] * len(tokenized_outputs)

        # Padding (Đệm) cho đủ max_length
        padding_length = self.max_length - len(tokenized_outputs)
        if padding_length > 0:
            tokenized_outputs = tokenized_outputs + [self.tokenizer.pad_token_id] * padding_length
            label_ids = label_ids + [-100] * padding_length
            attention_mask = attention_mask + [0] * padding_length

        item = {
            "input_ids": torch.tensor(tokenized_outputs, dtype=torch.long),
            "attention_mask": torch.tensor(attention_mask, dtype=torch.long),
            "labels": torch.tensor(label_ids, dtype=torch.long)
        }
        return item

# 1. Khởi tạo Tokenizer
tokenizer = AutoTokenizer.from_pretrained("vinai/phobert-base-v2")

# 2. Khởi tạo lại Dataset
train_dataset = ViMedNERDataset(train_sentences, tokenizer, label2id)
dev_dataset = ViMedNERDataset(dev_sentences, tokenizer, label2id)

config.json:   0%|          | 0.00/678 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/895k [00:00<?, ?B/s]

bpe.codes:   0%|          | 0.00/1.14M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/3.13M [00:00<?, ?B/s]

- gán nhãn -100 cho các sub_token đằng sau phần được tách ra đầu tiên của từ có nhãn, khi huấn luyện model tự biến để bỏ qua khi tính loss tránh học 2 sub_token của cùng 1 từ cần nhận biết
- attention mask sẽ giúp tạo ra những giá trị 0 cho seft-attention khi tính để bỏ qua attention của pad, cls, sep

In [ ]:
# Xem thử 2 mẫu đầu tiên trong train_dataset
for i in range(2):
    sample = train_dataset[i]
    print(f"--- Mẫu số {i} ---")
    for key, val in sample.items():
        print(f"{key}: {val.shape}")

--- Mẫu số 0 ---
input_ids: torch.Size([128])
attention_mask: torch.Size([128])
labels: torch.Size([128])
--- Mẫu số 1 ---
input_ids: torch.Size([128])
attention_mask: torch.Size([128])
labels: torch.Size([128])


In [ ]:
from transformers import AutoModelForTokenClassification, TrainingArguments, Trainer, DataCollatorForTokenClassification
import warnings
warnings.filterwarnings('ignore', category=UserWarning, module='seqeval')
# 1. Khởi tạo mô hình PhoBERT
model = AutoModelForTokenClassification.from_pretrained(
    "vinai/phobert-base-v2",
    num_labels=len(label2id),
    id2label=id2label,
    label2id=label2id
)

# 2. Tạo DataCollator
data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)

# 3. Cấu hình tham số huấn luyện
training_args = TrainingArguments(
    output_dir="./vimedner_baseline",
    eval_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=5,
    weight_decay=0.01,
    logging_steps=50,
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    report_to="none"
)

# 4. Khởi tạo Trainer (đã lược bỏ dòng truyền tokenizer vào đây)
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=dev_dataset,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

# 5. Tiến hành huấn luyện Baseline
trainer.train()

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaForTokenClassification LOAD REPORT from: vinai/phobert-base-v2
Key                       | Status     | 
--------------------------+------------+-
lm_head.bias              | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
classifier.bias           | MISSING    | 
classifier.weight         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Accuracy
1,0.560534,0.464422,0.586962,0.611544,0.599001,0.883114
2,0.413455,0.393071,0.620482,0.663624,0.641328,0.894919
3,0.352564,0.363878,0.622618,0.684295,0.652002,0.896468
4,0.293446,0.365890,0.588515,0.698792,0.638930,0.889649
5,0.260711,0.363578,0.609321,0.705503,0.653894,0.894117


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=1430, training_loss=0.4194902006562773, metrics={'train_runtime': 677.3895, 'train_samples_per_second': 33.755, 'train_steps_per_second': 2.111, 'total_flos': 1493759120666880.0, 'train_loss': 0.4194902006562773, 'epoch': 5.0})

In [ ]:
from transformers import pipeline

# Tạo ống kính (pipeline) nhận mô hình tốt nhất của bạn
nlp_ner = pipeline(
    "token-classification",
    model=model,
    tokenizer=tokenizer,
    device=model.device
)

# Test thử luôn với câu của bạn
text = "Bệnh nhân có tiền sử cao huyết áp và đái tháo đường tuýp 2 đang điều trị sau đó không chữa được nên dẫn đến run tay chân, tuyến tiền liệt bị chảy máu."
results = nlp_ner(text)

for item in results:
    print(f"{item['word']}: {item['entity']}")

cao: B-ten_benh
huyết: I-ten_benh
áp: I-ten_benh
đái: B-ten_benh
tháo: I-ten_benh
đường: I-ten_benh
tuýp: I-ten_benh
2: I-ten_benh
run: B-trieu_chung_benh
tay: I-trieu_chung_benh
châ@@: I-trieu_chung_benh
chảy: B-trieu_chung_benh
má@@: I-trieu_chung_benh


In [ ]:
import torch
from transformers import AutoModelForTokenClassification, AutoTokenizer, Trainer, TrainingArguments, DataCollatorForTokenClassification

#baseline_model train trên cả train + dev
# 1. Đường dẫn mô hình đã lưu
model_path = "/content/drive/MyDrive/vimedner_final_baseline"

# 2. Tải Tokenizer và Model
eval_tokenizer = AutoTokenizer.from_pretrained(model_path)
eval_model = AutoModelForTokenClassification.from_pretrained(model_path)

# 3. Chuẩn bị tập Test
test_sentences = load_conll_data('/content/drive/MyDrive/datasetViMedNER/testdata/test.txt') # Kiểm tra đúng thư mục testdata hoặc traindata
test_dataset = ViMedNERDataset(test_sentences, eval_tokenizer, label2id)
print(f"✅ Đã tải {len(test_dataset)} câu từ Test Set.")

# 4. Cấu hình đánh giá tối ưu tốc độ
eval_args = TrainingArguments(
    output_dir="./eval_temp",
    per_device_eval_batch_size=32,   # Tăng batch size lên 32 để chạy nhanh gấp đôi
    report_to="none"
)

# 5. Khởi tạo Trainer dùng trực tiếp hàm compute_metrics đã khai báo
eval_trainer = Trainer(
    model=eval_model,
    args=eval_args,
    data_collator=DataCollatorForTokenClassification(tokenizer=eval_tokenizer),
    compute_metrics=compute_metrics  # Sử dụng trực tiếp hàm compute_metrics sẵn có
)

# 6. Chạy đánh giá và in kết quả
print("⏳ Đang tiến hành chấm điểm...")
test_results = eval_trainer.predict(test_dataset)

print("\n" + "="*35)
print("=== KẾT QUẢ ĐÁNH GIÁ TRÊN TEST SET ===")
print("="*35)
for metric, score in test_results.metrics.items():
    print(f"{metric:<25}: {score}")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

✅ Đã tải 1525 câu từ Test Set.
⏳ Đang tiến hành chấm điểm...



=== KẾT QUẢ ĐÁNH GIÁ TRÊN TEST SET ===
test_loss                : 0.3616098165512085
test_model_preparation_time: 0.0064
test_precision           : 0.6041002277904328
test_recall              : 0.7072
test_f1                  : 0.6515970515970516
test_accuracy            : 0.8940277752695357
test_runtime             : 789.6129
test_samples_per_second  : 1.931
test_steps_per_second    : 0.061
